# Multimodal MHIST Diagnostic Demo
This interactive notebook demonstrates the final **Top-1 Multimodal Finetuned QuiltNet** model. 
You can run the cells below to load the optimal model weights discovered during Bayesian Hyperparameter Optimization and see exactly how it processes histology images and clinical text to make a diagnosis.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import open_clip
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import ast
import sys
import os

# Add our source code directory to the path
sys.path.append(os.path.abspath('quilt_src'))
from multimodal_experiment import MultimodalQuiltClassifier
from visualize_attention import get_attention_map, get_text_attention_map

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 1. Load the Model and Dataset
We load the pre-trained `QuiltNet-B-32` backbone and our final finetuned classifier weights.

In [ ]:
# Initialize model configuration
MODEL_NAME = "hf-hub:wisdomik/QuiltNet-B-32"
base_model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=True)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

# Use the optimal dropout rate discovered via Optuna
classifier = MultimodalQuiltClassifier(base_model, dropout_rate=0.5110907127277374).to(device)

# Load the best saved checkpoint
best_model_path = "quilt_results/finetuned_top1/mhist_with_captions_all_prompts/best_study_model.pth"
classifier.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
classifier.eval()
print("Successfully loaded optimal model weights!")

# Load dataset annotations
df = pd.read_csv("captions/mhist_with_captions_all_prompts.csv")
annotations = pd.read_csv("quilt_data/annotations.csv")
df = df.merge(annotations, left_on="image", right_on="Image Name", how="inner")
print(f"Loaded {len(df)} patient cases.")

### 2. Interactive Visualization Function
This function takes a random patient case, passes the image and clinical text through our dual-encoder model, and extracts the exact internal Self-Attention maps to show you *why* it made its decision.

In [ ]:
def analyze_random_case(dataframe, target_class=None):
    if target_class:
        sample = dataframe[dataframe['Majority Vote Label'] == target_class].sample(1).iloc[0]
    else:
        sample = dataframe.sample(1).iloc[0]
        
    img_path = os.path.join("images", sample['image'])
    raw_img = Image.open(img_path).convert("RGB")
    
    # Preprocess
    img_tensor = preprocess(raw_img).unsqueeze(0).to(device)
    prompts = ast.literal_eval(sample['top_prompts'])
    top_prompt = prompts[0]
    text_tokens = tokenizer([top_prompt]).to(device)
    
    # Inference
    with torch.no_grad():
        outputs = classifier(img_tensor, text_tokens)
        probs = torch.softmax(outputs, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()
        pred_label = "HP" if pred_idx == 1 else "SSA"
        confidence = probs[0][pred_idx].item()
        
    true_label = sample.get('Majority Vote Label', 'Unknown')
    
    # Extract Interpretability Attention Maps
    attn_map = get_attention_map(classifier, img_tensor, device, suppress_sinks=True)[0]
    text_attn, words = get_text_attention_map(classifier, text_tokens, tokenizer)
    
    # --- Plotting ---
    plt.figure(figsize=(18, 5))
    
    # 1. Original Image
    plt.subplot(1, 3, 1)
    plt.imshow(raw_img)
    title_color = "black" if pred_label == true_label else "red"
    plt.title(f"True: {true_label} | Pred: {pred_label} ({confidence:.1%})
Prompt: {top_prompt}", color=title_color)
    plt.axis('off')
    
    # 2. Vision Attention Map
    plt.subplot(1, 3, 2)
    plt.imshow(raw_img)
    heatmap = F.interpolate(attn_map.unsqueeze(0).unsqueeze(0), size=raw_img.size[::-1], mode='bicubic').squeeze()
    heatmap_np = heatmap.numpy()
    heatmap_np = (heatmap_np - heatmap_np.min()) / (heatmap_np.max() - heatmap_np.min() + 1e-8)
    plt.imshow(heatmap_np, cmap='jet', alpha=0.5)
    plt.title("Vision Transformer Focus (Image Attention)")
    plt.axis('off')
    
    # 3. Text Attention Bar Chart
    ax3 = plt.subplot(1, 3, 3)
    valid_indices = [j for j, w in enumerate(words) if w.strip() not in ['.', ',', ';', ':', '!', '?', '(', ')', '-'] and w.strip() != '']
    filtered_words = [words[j] for j in valid_indices]
    filtered_text_attn = text_attn[valid_indices]
    
    text_attn_norm = (filtered_text_attn - filtered_text_attn.min()) / (filtered_text_attn.max() - filtered_text_attn.min() + 1e-8)
    colors = plt.cm.plasma(text_attn_norm)
    
    ax3.barh(np.arange(len(filtered_words)), filtered_text_attn, color=colors)
    ax3.set_yticks(np.arange(len(filtered_words)))
    ax3.set_yticklabels(filtered_words)
    ax3.invert_yaxis()
    ax3.set_title("Text Encoder Focus (Prompt Attention)")
    ax3.set_xlabel("Relative Attention Weight")
    
    plt.tight_layout()
    plt.show()

### 3. Run Predictions!
Run the cell below to randomly sample a **Sessile Serrated Adenoma (SSA)** case and visualize the model's diagnostic reasoning. You can rerun this cell as many times as you like!

In [ ]:
analyze_random_case(df, target_class="SSA")

Run the cell below to sample a **Hyperplastic Polyp (HP)** case.

In [ ]:
analyze_random_case(df, target_class="HP")